<a href="https://colab.research.google.com/github/puhspi04/myproject/blob/main/mobile_net_maindataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [10]:
import zipfile
import os

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [11]:
os.listdir("dataset")

['Main Dataset']

In [27]:
#1
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [28]:
#2 Image settings
img_size = 224
batch_size = 32


In [29]:
#3 Data generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [30]:
#4 Training data
train_data = datagen.flow_from_directory(
    "dataset/Main Dataset",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

Found 1248 images belonging to 6 classes.


In [31]:
#5 Validation data
val_data = datagen.flow_from_directory(
    "dataset/Main Dataset",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

Found 310 images belonging to 6 classes.


In [32]:
#6 Load MobileNetV2 (pretrained)
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)


In [33]:
#7 Freeze base layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers[-20:]:
    layer.trainable = True


In [40]:
#8 Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
predictions = Dense(train_data.num_classes, activation="softmax")(x)

In [41]:
#9 Create final model
model = Model(inputs=base_model.input, outputs=predictions)

In [42]:
#10 Compile
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 1,370,822 (5.23 MB)

 Non-trainable params: 1,051,904 (4.01 MB)

In [50]:
#11 Train model

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)


Epoch 1/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 103s 3s/step - accuracy: 0.9776 - loss: 0.0724 - val_accuracy: 0.6774 - val_loss: 7.2347
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.9744 - loss: 0.0976 - val_accuracy: 0.4903 - val_loss: 20.5879
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.9768 - loss: 0.0979 - val_accuracy: 0.7452 - val_loss: 7.9172
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.9808 - loss: 0.0864 - val_accuracy: 0.8000 - val_loss: 5.7079
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.9888 - loss: 0.0415 - val_accuracy: 0.8387 - val_loss: 3.4855
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.9936 - loss: 0.0294 - val_accuracy: 0.8161 - val_loss: 4.7405
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.9944 - loss: 0.0253 - val_accuracy: 0.8065 - val_loss: 5.0815
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 102s 3s/step - accuracy: 0.9928 - loss: 0.0304 - val_accuracy: 0.8290 - val_lo

In [48]:
#12 Save model
model.save("mobilenet_model.keras")
print("Model saved successfully!")


Model saved successfully!


In [51]:
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print(f"Final Training Accuracy: {train_acc:.4f}")
print(f"Final Validation Accuracy: {val_acc:.4f}")

Final Training Accuracy: 0.9896
Final Validation Accuracy: 0.8290
